In [22]:
import socket
print(socket.gethostname())

fc30341


In [ ]:
#import sys
#!{sys.executable} -m pip install pandas numpy matplotlib seaborn torch

In [2]:
import numpy as np
import torch

print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")
#print(f"NumPy C-API: {hex(np.lib.NumpyVersion(np.__version__)._version)}")

NumPy:  1.25.2
PyTorch: 2.0.1


In [3]:
import torch
import numpy as np
import scipy.linalg as linalg
import matplotlib.pyplot as plt
print("Hello")

Hello


In [4]:
x = torch.Tensor(5, 3)
print(x)
y = torch.rand(5, 3)
print(y)
# let us run the following only if CUDA is available
if torch.cuda.is_available():
    x = x.cuda()
    y = y.cuda()
    print("There is cuba")
    print(x + y)
print("Reach end")

tensor([[-7.6061e+27,  7.3947e-42,  3.2786e-34],
        [ 0.0000e+00,  1.3162e-07,  7.3947e-42],
        [ 1.3025e-07,  7.3947e-42, -2.8180e+26],
        [ 7.3947e-42,  1.3026e-07,  7.3947e-42],
        [-5.9432e+27,  7.3947e-42, -5.9432e+27]])
tensor([[0.8423, 0.7775, 0.7470],
        [0.3180, 0.6453, 0.2206],
        [0.3342, 0.8445, 0.7192],
        [0.5516, 0.5450, 0.8488],
        [0.2269, 0.9065, 0.6313]])
Reach end


In [5]:
def gaussian3d(x,mean,covar = np.array([[5,0,0],[0,5,0],[0,0,5]])):

    '''
    returns a 3d gaussian evaluated for an image and time location with a given covariance matrix and mean
    '''
    

    Ninv = np.linalg.pinv(covar)

    return np.exp(-np.diag((x-mean).T@Ninv@(x-mean))) ## there's probably a smarter way to do this
    #Normalized: 

    #return ((2*np.pi)**-1.5)*(np.linalg.det(covar)**-0.5)*np.exp(-0.5*np.diag((x-mean).T@Ninv@(x-mean)))


def genrandomfield(xlen,ylen,tlen,Ntrans,widths = None,noiseamp = 0.1):

    '''
    generates a field of white noise with a given number of overlaid gaussian 'transients'.
    Transients and noise returned as separate layers. 

    Arguements (in order):

    xlen: length of the first image dimension, in pixels
    ylen: same as above but for second dimension
    tlen: length of the image/noise fields to be generated, in time (arbitrary units)
    Ntrans: how many transients to simulate
    widths: a 3-tuple (A,B,C) of sigma values corresponding to the x, y and time widths of the 3D Gaussian transients.
    for example, widths = (1,2,10) will give transients that are narrow in the first image dimension, broader in the second,
    and have a characteristic timescale of about 10 time units.
    noiseamp: the root-mean-square level of (white) noise in the image.

    returns: (in order)
    combinedfield, noisefield+transient_field
    noisefield: the noise layer, with shape (xlen,ylen,tlen)
    transient_field: the transient layer, with shape (xlen,ylen,tlen)
    x0: the list of positions of each source in the first image axis
    y0: the list of positions of each source in the second image axis
    t0: the central times of each transient signal (time of peak brightness)
    '''

    x = np.arange(xlen) ## x and y arrays
    y = np.arange(ylen)

    xx,yy = np.meshgrid(x,y) ## create a grid of x,y positions to evaluate at

    xxf,yyf = xx.ravel(),yy.ravel() ## flatten for processing

    rng = np.random.default_rng(123)

    noisefield = noiseamp*rng.standard_normal(size = (xlen,ylen,tlen)) # generate the noise

    transient_field = np.zeros(noisefield.shape) ## array for the transient layer to be created in
    combined_field = np.zeros(noisefield.shape)

    x0 = [] ## these record the x, y, and time mean values for each transient signal
    y0 = [] ## these record the x, y, and time mean values for each transient signal
    t0 = [] ## these record the x, y, and time mean values for each transient signal

    posvecs = np.zeros((3,len(xxf)))
    posvecs[0,:] = xxf
    posvecs[1,:] = yyf

    if widths:
        covar = np.array([[widths[0]**2,0,0],[0,widths[1]**2,0],[0,0,widths[2]**2]])
    
    for i in range(Ntrans):

        x0.append(np.random.randint(xlen)) ## record where this trans happened
        y0.append(np.random.randint(ylen))
        t0.append(np.random.randint(tlen))

        for t in range(tlen):

            posvecs[2,:] = t

            if widths:

                trans = gaussian3d(posvecs, mean = np.array([[x0[-1],y0[-1],t0[-1]]]).T, covar = covar).reshape((xlen,ylen))
            else:

                trans = gaussian3d(posvecs, mean = np.array([[x0[-1],y0[-1],t0[-1]]]).T).reshape((xlen,ylen))

            transient_field[:,:,t] += trans
            combined_field[:,:,t] = transient_field[:,:,t] + noisefield[:,:,t]

    median = np.vstack((x0, y0, t0))
    return combined_field, noisefield,transient_field,median

In [ ]:
def genrandomfields(data_size = 1, Ntrans = 5, xlen=50, ylen=50, tlen=100):
    widths = (5, 5, 10)
    combined_fields = []
    medians = []
    for datum in range(data_size):
        combined, noise, transient, median = genrandomfield(xlen,ylen,tlen,Ntrans,widths)
        combined_fields.append(combined)
        medians.append(median)
    combined_fields = np.array(combined_fields)
    medians = np.array(medians)
    return combined_fields, medians

In [7]:
training,n_0,t_0,training_median = genrandomfield(50,50,100,5, widths = (5,5,10))#, noiseamp=1e-10)
#testing,n_1,t_1,testing_median = genrandomfield(50,50,100,5, widths = (5,5,10))#, noiseamp=1e-10)
print(training_median)

[[20 32 16  6 36]
 [ 1  8 10  6 44]
 [57 78 13 81 73]]


In [8]:
# generates and saves animation frames for the above generated field

for frame in range(t_0.shape[-1]):
    plt.figure(figsize = (5,5))
    plt.imshow(training[:,:,frame],vmax = 1,vmin = 0)
    plt.xticks([])
    plt.yticks([])
    #plt.show()
    #plt.savefig('example_animation/frame_%d.png' %frame)
    plt.close()

In [9]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
#from torchvision import datasets, transforms
print("Os imported")
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
#device = "mps"
print(f"Using {device} device")

Os imported
Using cpu device


In [10]:
X, y = genrandomfields(64, 5, 50, 50, 100)
print("Training set gained")

Training set gained


In [ ]:
def convert_to_torch_data(data):
    data = torch.tensor(data, dtype=torch.float32)
    data = data.squeeze(0)
    #data = data.permute(0, 3, 1, 2)
    data = data.float().to(device)
    return data

In [ ]:
print(X.shape)
X_train = convert_to_torch_data(X)
X_train = X_train.permute(0, 3, 1, 2)
y_train = convert_to_torch_data(y)
print(X_train.shape)
X_train.to(device)
y_train.to(device)
y_train = y_train[:, :, 0]
X, y = genrandomfields(10, 5, 50, 50, 100)
X_test = convert_to_torch_data(X)
X_test = X_test.permute(0, 3, 1, 2)
y_test = convert_to_torch_data(y)
X_test.to(device)
y_test.to(device)
y_test = y_test[:, :, 0]
y_train.to(torch.long)
y_test.to(torch.long)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(y_test.dtype)


(64, 50, 50, 100)
torch.Size([64, 100, 50, 50])
torch.Size([64, 100, 50, 50])
torch.Size([64, 3])
torch.Size([10, 100, 50, 50])
torch.Size([10, 3])
torch.float32


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# 1. Wrap tensors in TensorDataset
train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test,  y_test)

# 2. Create DataLoaders
batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(50*50*100, 250),
            nn.ReLU(),
            nn.Linear(250, 250),
            nn.ReLU(),
            nn.Linear(250, 3),
        )

    def forward(self, x):
        #x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [15]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=250000, out_features=250, bias=True)
    (2): ReLU()
    (3): Linear(in_features=250, out_features=250, bias=True)
    (4): ReLU()
    (5): Linear(in_features=250, out_features=3, bias=True)
  )
)


In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    all_preds = []

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            #print("The pred shape is: ", pred.shape)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y.argmax(1)).type(torch.float).sum().item()
            all_preds.append(pred)
            #print("The y shape is:", y.shape)

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    all_preds = torch.cat(all_preds, dim=0)  
    return all_preds

In [17]:
learning_rate = 1e-3
batch_size = 64
epochs = 100
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
preds = test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 108.863525  [   64/   64]
Test Error: 
 Accuracy: 70.0%, Avg loss: 104.520775 

Epoch 2
-------------------------------
loss: 101.969254  [   64/   64]
Test Error: 
 Accuracy: 70.0%, Avg loss: 104.427734 

Epoch 3
-------------------------------
loss: 101.560837  [   64/   64]
Test Error: 
 Accuracy: 70.0%, Avg loss: 106.792923 

Epoch 4
-------------------------------
loss: 102.542290  [   64/   64]
Test Error: 
 Accuracy: 70.0%, Avg loss: 106.553932 

Epoch 5
-------------------------------
loss: 104.015381  [   64/   64]
Test Error: 
 Accuracy: 70.0%, Avg loss: 109.525345 

Epoch 6
-------------------------------
loss: 105.149139  [   64/   64]
Test Error: 
 Accuracy: 70.0%, Avg loss: 104.332504 

Epoch 7
-------------------------------
loss: 100.657990  [   64/   64]
Test Error: 
 Accuracy: 70.0%, Avg loss: 104.464828 

Epoch 8
-------------------------------
loss: 100.182068  [   64/   64]
Test Error: 
 Accuracy: 70.0%, Avg loss: 104.2

In [23]:
print(preds)
print(X_test)
print(y_test)

tensor([[ 0.1129, -0.9323,  0.7710],
        [ 0.1194,  0.0220, -0.0250],
        [-0.4649, -0.2762,  0.9124],
        [-0.3086,  0.0424,  0.4731],
        [ 0.0536,  0.0442,  0.1187],
        [ 0.0859,  0.3017, -0.1649],
        [ 0.2429, -0.5515,  0.3919],
        [-0.4284,  0.2642,  0.4398],
        [ 0.4015, -0.1268, -0.1424],
        [ 0.1796, -0.6193,  0.5178]])
tensor([[[[-0.0989, -0.0516, -0.0937,  ...,  0.0472, -0.1639, -0.0560],
          [ 0.1459, -0.0067, -0.0071,  ...,  0.0024, -0.0021,  0.1544],
          [-0.0472,  0.0420,  0.2439,  ...,  0.1764, -0.0921, -0.0926],
          ...,
          [-0.0783,  0.0673, -0.0472,  ...,  0.0011, -0.0082, -0.0748],
          [ 0.1427, -0.1376,  0.0399,  ...,  0.2144, -0.0752,  0.1259],
          [-0.1910, -0.0215, -0.1084,  ..., -0.0558,  0.0903, -0.1717]],

         [[-0.0368,  0.1658, -0.0809,  ...,  0.0593, -0.0010, -0.1128],
          [ 0.0104,  0.1079,  0.0401,  ..., -0.0719,  0.0608,  0.1706],
          [-0.1664, -0.0084,  0.1323

In [19]:
'''
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")
'''

'\nprint(f"Model structure: {model}\n\n")\n\nfor name, param in model.named_parameters():\n    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")\n'

In [20]:
#custom dataset
'''
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) 

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
'''

'\nfrom torch.utils.data import Dataset, DataLoader\n\nclass CustomDataset(Dataset):\n    def __init__(self, X, y):\n        self.X = torch.tensor(X, dtype=torch.float32)\n        self.y = torch.tensor(y, dtype=torch.long) \n\n    def __len__(self):\n        return len(self.X)\n\n    def __getitem__(self, idx):\n        return self.X[idx], self.y[idx]\n'

In [21]:
'''
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 50
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")
'''

'\nloss_fn = nn.CrossEntropyLoss()\noptimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)\n\nepochs = 50\nfor t in range(epochs):\n    print(f"Epoch {t+1}\n-------------------------------")\n    train_loop(train_dataloader, model, loss_fn, optimizer)\n    test_loop(test_dataloader, model, loss_fn)\nprint("Done!")\n'